# Customer Intelligence & E-Commerce Analytics Suite
### End-to-End Behavioral Segmentation, Churn Modeling, Market Basket Mining & Lifetime Value

Welcome to the **Customer Intelligence Platform** deep-dive notebook. This notebook walks through production-grade workflows for:
1. **Exploratory Data Analysis (EDA)** on transactional data
2. **RFM Segmentation** (Heuristic & K-Means Clustering)
3. **Predictive Machine Learning Churn Classification**
4. **Market Basket Analysis (MBA)** & Product Association Rules
5. **Discounted Customer Lifetime Value (CLV)** Modeling

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import generate_synthetic_data, clean_and_preprocess_data
from src.rfm_analysis import compute_rfm_metrics, score_rfm_quintiles, map_rfm_segments, apply_kmeans_clustering, generate_segment_summary
from src.churn_model import engineer_churn_features, train_and_evaluate_models
from src.basket_analysis import extract_transaction_baskets, generate_association_rules
from src.clv_calculator import calculate_customer_lifetime_value

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('Libraries and modules successfully imported!')

## 1. Data Ingestion & Preprocessing
We load cleaned e-commerce transaction logs with order quantities, unit prices, product categories, and customer demographics.

In [ ]:
data_path = ROOT_DIR / 'data' / 'processed' / 'cleaned_transactions.parquet'
if data_path.exists():
    df = pd.read_parquet(data_path)
else:
    raw = generate_synthetic_data(num_customers=1500, seed=42)
    df = clean_and_preprocess_data(raw)

print(f'Total records: {len(df):,}')
print(f'Unique customers: {df["CustomerID"].nunique():,}')
print(f'Unique orders: {df["InvoiceNo"].nunique():,}')
df.head()

## 2. RFM Customer Segmentation
Recency, Frequency, and Monetary scores segment accounts into strategic business tiers.

In [ ]:
rfm = compute_rfm_metrics(df)
scored_rfm = score_rfm_quintiles(rfm)
segmented_rfm = map_rfm_segments(scored_rfm)
clustered_rfm, kmeans_model = apply_kmeans_clustering(segmented_rfm, n_clusters=4)

summary = generate_segment_summary(clustered_rfm)
display(summary)

plt.figure(figsize=(10, 4))
sns.barplot(data=summary, x='TotalRevenue', y='Segment', palette='Blues_r')
plt.title('Total Revenue Contribution by Customer Segment ($)', fontsize=14, fontweight='bold')
plt.xlabel('Revenue ($)')
plt.show()

## 3. Supervised Machine Learning Churn Modeling
We train and benchmark Logistic Regression, Random Forest, and Gradient Boosting to identify accounts at risk of dropping off.

In [ ]:
churn_features = engineer_churn_features(df, churn_inactivity_days=120)
model_results, predictions = train_and_evaluate_models(churn_features)

print('Best Performing Model:', model_results['best_model'])
for model_name in ['Random Forest', 'Gradient Boosting', 'Logistic Regression']:
    res = model_results[model_name]
    print(f"{model_name:20} -> ROC-AUC: {res['roc_auc']:.4f} | F1: {res['f1']:.4f} | Accuracy: {res['accuracy']:.4f}")

# Feature Importance Plot
feat_imp = pd.Series(model_results['feature_importances'])
plt.figure(figsize=(9, 4))
feat_imp.sort_values().plot(kind='barh', color='#2563eb')
plt.title('Key Drivers of Customer Churn (Feature Importance)', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.show()

## 4. Market Basket Analysis & Product Affinity
Using association rule mining, we discover high-lift co-purchased bundles to drive cross-sells.

In [ ]:
baskets = extract_transaction_baskets(df)
rules = generate_association_rules(baskets, min_support=0.01, min_confidence=0.15, min_lift=1.05)

print(f'Total rules mined: {len(rules)}')
display(rules.head(10))

if not rules.empty:
    plt.figure(figsize=(8, 4))
    sns.scatterplot(data=rules, x='Support', y='Confidence', size='Lift', hue='Lift', palette='plasma', sizes=(40, 200))
    plt.title('Market Basket Association Rules (Support vs. Confidence)', fontsize=13, fontweight='bold')
    plt.show()

## 5. Discounted Customer Lifetime Value (CLV)
Multi-period forecasting factoring in purchase cadence, profit margins, and survival rates.

In [ ]:
clv = calculate_customer_lifetime_value(segmented_rfm, predictions, profit_margin=0.25, discount_rate=0.10)
display(clv[['CustomerID', 'Frequency', 'Monetary', 'HistoricalProfit', 'PredictiveCLV_2Year', 'TotalEstimatedCLV', 'CLV_Tier']].head(10))

tier_summary = clv.groupby('CLV_Tier')['TotalEstimatedCLV'].agg(['count', 'mean', 'sum']).reset_index()
display(tier_summary)